# LAB - IPM y ICS por manzana (Ciudad Paraiso)

Mapa de **laboratorio** para visualizar de forma simple la pobreza (IPM) y la condicion
social (ICS) por manzana. El panel usa **medidores legibles** (no una matriz tecnica) para
que cualquier persona entienda la informacion; al tocar una manzana, muestra su detalle.

- Fuente oficial: `data/geojson/IPM_ciudad_Paraiso.geojson` (33 manzanas; trae `ipm`,
  `ics` y las 15 variables de privacion).
- **IPM** (pobreza, inverso: mayor = peor) en paleta viridis; **ICS** (condicion social,
  directo: mayor = mejor) en paleta cividis. Escala oficial de 8 categorias.
- Panel: dos medidores (IPM y ICS) con su categoria en palabras y comparados con el
  promedio de la zona + lectura automatica + barras de privaciones de la manzana tocada.


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
if _en_colab():
    import os, subprocess, sys, zipfile
    REPO = "https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git"
    DEST = "/content/mapa_Analitico_ciudad_Paraiso"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "geopandas", "folium", "mapclassify", "pyarrow", "openpyxl"], check=False)
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=False)
    _zip = os.path.join(DEST, "data_publico", "data_itt_publico.zip")
    _data_dir = os.path.join(DEST, "data")
    if os.path.isfile(_zip):
        os.makedirs(_data_dir, exist_ok=True)
        with zipfile.ZipFile(_zip) as z:
            z.extractall(_data_dir)
        print("Datos publicos descomprimidos en:", os.path.join(_data_dir, "1_data_itt"))
    else:
        print("Aviso: no se encontro", _zip, "- revisa que el repo lo incluya.")
    _nb_dir = os.path.join(DEST, "notebooks")
    if os.path.isdir(_nb_dir):
        os.chdir(_nb_dir)
    print("Entorno Colab listo. Directorio de trabajo:", os.getcwd())
else:
    print("Entorno local (uv). Setup Colab omitido.")

## 1. Entorno y paleta Okabe-Ito

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"
OKI_AMARILLO="#F0E442"; OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"
OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"
CRS_WEB=4326; CRS_METRIC=9377
print("Entorno listo.")

## 2. Rutas y poligono de la zona

In [ ]:
def _resolver_poligono():
    env=os.environ.get("POLIGONO_ZONA"); cand=[env] if env else []
    cand+=[os.path.join("..","data","geojson","poligono_ciudad_Paraiso.geojson"),
           os.path.join("data","geojson","poligono_ciudad_Paraiso.geojson"),
           "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
           "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson"]
    for p in cand:
        if p and os.path.isfile(p): return os.path.abspath(p)
    return None
_pp=_resolver_poligono()
POLIGONO_ZONA=gpd.read_file(_pp).to_crs(CRS_WEB) if _pp else None
print("Poligono:", _pp)
OUT_DIR=None
for _b in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_b,exist_ok=True); OUT_DIR=os.path.abspath(_b); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Salida de mapas HTML:", MAPAS_DIR)
def _ruta_itt(n):
    for p in [os.path.join("..","data","1_data_itt",n),os.path.join("data","1_data_itt",n),
              os.path.join("/content","data","1_data_itt",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","1_data_itt",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
def _ruta_geojson(n):
    for p in [os.path.join("..","data","geojson",n),os.path.join("data","geojson",n),
              os.path.join("/content","data","geojson",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","geojson",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None

## 3. Carga de la fuente oficial IPM + ICS

In [ ]:
# 3a. Carga de la fuente oficial IPM+ICS (una sola capa trae ipm, ics y las 15 variables).
_rp = _ruta_geojson("IPM_ciudad_Paraiso.geojson")
if _rp is None:
    raise FileNotFoundError("No se encontro IPM_ciudad_Paraiso.geojson en data/geojson.")
g = gpd.read_file(_rp)
g = g.set_crs(epsg=CRS_WEB) if g.crs is None else g.to_crs(CRS_WEB)
g = g[g.geometry.notna()].copy()
g["ipm"] = pd.to_numeric(g["ipm"], errors="coerce")
g["ics"] = pd.to_numeric(g["ics"], errors="coerce")
# Recorte a la zona (por si la fuente trajera manzanas vecinas).
if POLIGONO_ZONA is not None:
    _z = POLIGONO_ZONA.geometry.union_all()
    g = g[g.geometry.within(_z) | g.geometry.intersects(_z)].copy()
g = g.reset_index(drop=True)
g["mz_id"] = range(len(g))
# Variables de privacion (etiquetas legibles) agrupadas por dimension.
VAR_LABEL = {
    "ANALF_": "Analfabetismo", "BAJO_": "Bajo logro educativo",
    "INFANCIA_": "Primera infancia", "INASIS_": "Inasistencia escolar",
    "REZAGO_": "Rezago escolar", "TRAB_INFAN": "Trabajo infantil",
    "DEPEN_": "Dependencia economica", "INFOR_": "Empleo informal",
    "SALUD_": "Barreras en salud", "ASEGU_": "Sin aseguramiento",
    "HACI_": "Hacinamiento", "PARED_": "Paredes inadecuadas",
    "EXCRE_": "Sin alcantarillado", "PISOS_": "Pisos inadecuados", "AGUA_": "Sin agua",
}
VARS_PRIV = [v for v in VAR_LABEL if v in g.columns]
for v in VARS_PRIV:
    g[v] = pd.to_numeric(g[v], errors="coerce").fillna(0.0)
IPM_PROM = round(float(g["ipm"].mean()), 1)
ICS_PROM = round(float(g["ics"].mean()), 1)
PRIV_PROM = {VAR_LABEL[v]: round(float(g[v].mean()), 1) for v in VARS_PRIV}
print("Manzanas IPM/ICS en zona:", len(g))
print("IPM promedio:", IPM_PROM, "| ICS promedio:", ICS_PROM)
ipm_z = g  # alias para el resto del notebook

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa interactivo (Folium)

In [ ]:
# 5a. Mapa laboratorio IPM/ICS: coropletas por manzana (IPM viridis, ICS cividis) con la
#      escala oficial de 8 categorias. Ambas clicables; alimentan el panel legible.
import folium
# Escala oficial de 8 categorias (misma para IPM e ICS; paletas distintas).
ESCALA = [
    (0.4, 20.0, "Bajo"), (20.1, 27.0, "Moderado-Bajo"), (27.1, 33.1, "Moderado"),
    (33.2, 39.8, "Moderado-Alto"), (39.9, 48.6, "Alto"), (48.7, 60.6, "Muy Alto"),
    (60.7, 80.0, "Extremo"), (80.1, 335.1, "Extremo Maximo"),
]
# Paletas de 8 pasos aptas daltonismo.
VIRIDIS = ["#440154", "#472d7b", "#3b528b", "#2c728e", "#21918c", "#28ae80", "#5ec962", "#fde725"]
CIVIDIS = ["#00224e", "#213d6b", "#555b6c", "#7d7c78", "#a59c74", "#c3b55f", "#e1cc48", "#fee838"]
def _cat_idx(v):
    v = float(v)
    for i, (lo, hi, _nom) in enumerate(ESCALA):
        if v <= hi:
            return i
    return len(ESCALA) - 1
def _cat_nombre(v):
    return ESCALA[_cat_idx(v)][2]
def _col_ipm(v):   # IPM: viridis, mayor = peor (color claro = mas pobreza)
    return VIRIDIS[_cat_idx(v)]
def _col_ics(v):   # ICS: cividis, mayor = mejor
    return CIVIDIS[_cat_idx(v)]
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro = [_c.y, _c.x]
else:
    centro = [g.geometry.union_all().centroid.y, g.geometry.union_all().centroid.x]
m = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
               zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m.get_root().width = "100%"; m.get_root().height = "100%"
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
if POLIGONO_ZONA is not None:
    _zg = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]
    _ag = []
    for _gm in (list(_zg.geoms) if _zg.geom_type == "MultiPolygon" else [_zg]):
        _ag.append([[x, y] for x, y in _gm.exterior.coords])
    folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
        name="Mascara (foco en la zona)", control=False,
        style_function=lambda f: {"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.2}).add_to(m)
_rp_lim = _ruta_itt("00_limite_ciudad_paraiso.geojson")
if _rp_lim is not None:
    _lim = gpd.read_file(_rp_lim).to_crs(CRS_WEB)
    _fgl = folium.FeatureGroup(name="Limite Ciudad Paraiso", show=True)
    folium.GeoJson(_lim.to_json(),
        style_function=lambda f: {"color":"#8A6D1A","weight":5,"dashArray":"10 6","fill":False}).add_to(_fgl)
    _fgl.add_to(m)
# Datos por manzana para el panel JS.
MZ_DATA = []
for _, r in g.iterrows():
    _priv = {VAR_LABEL[v]: round(float(r[v]), 1) for v in VARS_PRIV}
    MZ_DATA.append({
        "id": int(r["mz_id"]),
        "ipm": round(float(r["ipm"]), 1), "ics": round(float(r["ics"]), 1),
        "cat_ipm": _cat_nombre(r["ipm"]), "cat_ics": _cat_nombre(r["ics"]),
        "priv": _priv,
    })
# Capa IPM (viridis), activa; capa ICS (cividis), apagada. Ambas clicables.
def _add_coropleta(nombre, campo, colfn, show):
    _fg = folium.FeatureGroup(name=nombre, show=show)
    for _, r in g.iterrows():
        _v = float(r[campo]); _col = colfn(_v); _mid = int(r["mz_id"])
        _gj = folium.GeoJson(
            r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_col: {"fillColor": c, "color": "#ffffff",
                                               "weight": 0.6, "fillOpacity": 0.72}),
            highlight_function=lambda f: {"weight": 2.5, "color": "#111", "fillOpacity": 0.9},
            tooltip="%s: %.1f (%s)" % (campo.upper(), _v, _cat_nombre(_v)),
        )
        _gj.add_child(folium.Popup("mz:" + str(_mid), show=False))
        _gj.add_to(_fg)
    _fg.add_to(m)
    return _fg
_add_coropleta("IPM: pobreza por manzana (" + str(len(g)) + ")", "ipm", _col_ipm, True)
_add_coropleta("ICS: condicion social por manzana (" + str(len(g)) + ")", "ics", _col_ics, False)
if POLIGONO_ZONA is not None:
    m.fit_bounds([[3.442445,-76.53259],[3.44791,-76.526004]], padding=(30,30))
# Leyenda de paleta (IPM viridis + ICS cividis), escala de 8 categorias.
def _bloque_leyenda(titulo, paleta):
    _it = ""
    for (lo, hi, nom), col in zip(ESCALA, paleta):
        _it += ('<div style="display:flex;align-items:center;gap:5px;margin:1px 0">'
                '<i style="width:15px;height:11px;background:%s;display:inline-block;border:1px solid #999"></i>'
                '<span style="font-size:9px">%s (%s-%s)</span></div>'
                % (col, nom, ("%.0f" % lo).replace(".", ","), ("%.0f" % hi).replace(".", ",")))
    return ('<div style="margin-bottom:6px"><div style="font-weight:700;font-size:10px;margin-bottom:2px">%s</div>%s</div>'
            % (titulo, _it))
_leyenda = (
    '<div id="lab-leyenda" style="position:fixed;bottom:24px;right:24px;z-index:9999;'
    'background:' + BG + ';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    'font-family:system-ui,Segoe UI,sans-serif;color:' + OKI_GRIS + ';max-width:220px;max-height:60vh;overflow:auto">'
    + _bloque_leyenda("IPM - Pobreza (viridis, mayor=peor)", VIRIDIS)
    + _bloque_leyenda("ICS - Condicion social (cividis, mayor=mejor)", CIVIDIS)
    + '</div>')
m.get_root().html.add_child(folium.Element(_leyenda))
folium.LayerControl(collapsed=False, position="topright").add_to(m)
print("Mapa laboratorio IPM/ICS construido. Manzanas:", len(MZ_DATA))

## 6. Panel legible (medidores IPM/ICS + privaciones al tocar manzana)

In [ ]:
# 6a. Panel legible IPM/ICS: dos medidores (IPM pobreza, ICS condicion social) con su
#      categoria en palabras, comparados con el promedio de la zona, + barras de
#      privaciones de la manzana al tocarla. Lenguaje llano, sin jerga.
_MAP_ID = m.get_name()
_PAYLOAD = json.dumps({
    "manzanas": MZ_DATA,
    "ipm_prom": IPM_PROM, "ics_prom": ICS_PROM,
    "priv_prom": PRIV_PROM,
    "escala": [{"lo": lo, "hi": hi, "nom": nom} for (lo, hi, nom) in ESCALA],
}, ensure_ascii=False)

_panel = """
<div id="itt-stat-panel" style="position:fixed;top:16px;left:16px;z-index:9999;
     background:__BG__;width:min(420px,42vw);max-height:92vh;overflow:auto;
     padding:14px 16px;border-radius:12px;box-shadow:0 2px 12px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:13px;color:__GRIS__">
  <div style="font-weight:700;font-size:15px;margin-bottom:2px">Pobreza y condicion social</div>
  <div id="lab-sub" style="font-size:11px;color:#666;margin-bottom:10px">Promedio de Ciudad Paraiso. Toca una manzana en el mapa para ver su detalle.</div>

  <!-- Medidor IPM -->
  <div style="margin-bottom:12px">
    <div style="display:flex;justify-content:space-between;align-items:baseline">
      <span style="font-weight:700">IPM - Pobreza</span>
      <span><b id="lab-ipm-val">-</b> / 100</span>
    </div>
    <div style="height:14px;background:#e6e6e6;border-radius:7px;overflow:hidden;margin:3px 0">
      <div id="lab-ipm-bar" style="height:100%;width:0;background:#4c556c;transition:width .3s"></div>
    </div>
    <div style="display:flex;justify-content:space-between;font-size:11px;color:#666">
      <span id="lab-ipm-cat">-</span>
      <span>Mayor = mas pobreza</span>
    </div>
    <div id="lab-ipm-cmp" style="font-size:10px;color:#888"></div>
  </div>

  <!-- Medidor ICS -->
  <div style="margin-bottom:10px">
    <div style="display:flex;justify-content:space-between;align-items:baseline">
      <span style="font-weight:700">ICS - Condicion social</span>
      <span><b id="lab-ics-val">-</b> / 100</span>
    </div>
    <div style="height:14px;background:#e6e6e6;border-radius:7px;overflow:hidden;margin:3px 0">
      <div id="lab-ics-bar" style="height:100%;width:0;background:#a59c74;transition:width .3s"></div>
    </div>
    <div style="display:flex;justify-content:space-between;font-size:11px;color:#666">
      <span id="lab-ics-cat">-</span>
      <span>Mayor = mejor condicion</span>
    </div>
    <div id="lab-ics-cmp" style="font-size:10px;color:#888"></div>
  </div>

  <div id="lab-lectura" style="font-size:12px;background:#eef1f5;border-radius:8px;
       padding:8px 10px;margin:8px 0;line-height:1.35"></div>

  <div id="lab-priv-box" style="display:none">
    <div style="font-weight:700;font-size:12px;margin:6px 0 2px">Que le falta a esta manzana</div>
    <div style="font-size:10px;color:#666;margin-bottom:4px">% de hogares con cada privacion (de mayor a menor)</div>
    <div style="position:relative;height:230px;width:100%"><canvas id="lab-priv-chart"></canvas></div>
  </div>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var P = __PAYLOAD__;
  var MAP = "__MAP_ID__";
  var privChart = null;
  // Color del medidor segun categoria (misma escala; tono neutro legible).
  var CAT_COL = {"Bajo":"#2c7bb6","Moderado-Bajo":"#5aa6c9","Moderado":"#9cc9b0",
    "Moderado-Alto":"#e0c070","Alto":"#e8945a","Muy Alto":"#d7604a","Extremo":"#b2182b","Extremo Maximo":"#7a0177"};
  function catDe(v){
    for(var i=0;i<P.escala.length;i++){ if(v<=P.escala[i].hi) return P.escala[i].nom; }
    return P.escala[P.escala.length-1].nom;
  }
  function setMedidor(pref, val, cat){
    document.getElementById(pref+'-val').textContent = val.toFixed(1).replace('.',',');
    document.getElementById(pref+'-cat').textContent = cat;
    var bar=document.getElementById(pref+'-bar');
    bar.style.width = Math.max(2, Math.min(100, val))+'%';
    bar.style.background = CAT_COL[cat] || '#4c556c';
  }
  function cmpTxt(val, prom, inverso){
    var d = val - prom;
    if(Math.abs(d) < 0.5) return 'Igual que el promedio de la zona ('+prom.toFixed(1).replace('.',',')+')';
    var mas = d > 0;
    // inverso=true (IPM): mas alto = peor
    var peor = inverso ? mas : !mas;
    var signo = (mas?'+':'')+d.toFixed(1).replace('.',',');
    return (mas?'Por encima':'Por debajo')+' del promedio ('+prom.toFixed(1).replace('.',',')+', '+signo+') - '+
           (peor?'situacion mas desfavorable':'situacion mas favorable');
  }
  function lecturaZona(){
    document.getElementById('lab-sub').textContent='Promedio de Ciudad Paraiso. Toca una manzana en el mapa para ver su detalle.';
    setMedidor('lab-ipm', P.ipm_prom, catDe(P.ipm_prom));
    setMedidor('lab-ics', P.ics_prom, catDe(P.ics_prom));
    document.getElementById('lab-ipm-cmp').textContent='';
    document.getElementById('lab-ics-cmp').textContent='';
    document.getElementById('lab-lectura').innerHTML=
      'En promedio, la zona tiene pobreza <b>'+catDe(P.ipm_prom)+'</b> y condicion social <b>'+catDe(P.ics_prom)+'</b>.';
    document.getElementById('lab-priv-box').style.display='none';
  }
  function barsPriv(mz){
    var cv=document.getElementById('lab-priv-chart'); if(!cv||typeof Chart==='undefined')return;
    var arr=Object.keys(mz.priv).map(function(k){return [k,mz.priv[k]];})
             .sort(function(a,b){return b[1]-a[1];});
    if(privChart) privChart.destroy();
    privChart=new Chart(cv.getContext('2d'),{type:'bar',
      data:{labels:arr.map(function(x){return x[0];}),
            datasets:[{data:arr.map(function(x){return x[1];}),
              backgroundColor:'#3b528b',borderColor:'#21918c',borderWidth:0.5}]},
      options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,
        plugins:{legend:{display:false},tooltip:{callbacks:{label:function(c){return c.parsed.x+'%';}}}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:9}}},y:{ticks:{font:{size:9}}}}}});
  }
  function seleccionar(id){
    var mz=P.manzanas.find(function(m){return m.id===id;}); if(!mz) return;
    document.getElementById('lab-sub').textContent='Manzana seleccionada (toca otra para cambiar).';
    setMedidor('lab-ipm', mz.ipm, mz.cat_ipm);
    setMedidor('lab-ics', mz.ics, mz.cat_ics);
    document.getElementById('lab-ipm-cmp').textContent=cmpTxt(mz.ipm, P.ipm_prom, true);
    document.getElementById('lab-ics-cmp').textContent=cmpTxt(mz.ics, P.ics_prom, false);
    // Lectura automatica en lenguaje llano.
    var pobre = mz.ipm >= 33.3;
    var msg = 'Esta manzana tiene <b>pobreza '+mz.cat_ipm+'</b> y <b>condicion social '+mz.cat_ics+'</b>. ';
    if(pobre) msg += 'Supera el umbral de pobreza (33,3): zona a priorizar.';
    else msg += 'Esta por debajo del umbral de pobreza (33,3).';
    document.getElementById('lab-lectura').innerHTML=msg;
    document.getElementById('lab-priv-box').style.display='block';
    barsPriv(mz);
  }
  function hook(){
    var map=window[MAP]; if(!map){ setTimeout(hook,300); return; }
    map.on('popupopen', function(e){
      try{ var t=e.popup.getContent();
        if(typeof t==='string' && t.indexOf('mz:')===0){
          if(e.popup.remove) e.popup.remove();
          seleccionar(parseInt(t.split(':')[1],10));
        }
      }catch(err){}
    });
  }
  function init(){ lecturaZona(); hook(); }
  if(document.readyState==='complete'){ setTimeout(init,500); }
  else { window.addEventListener('load',function(){ setTimeout(init,500); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__PAYLOAD__", _PAYLOAD).replace("__MAP_ID__", _MAP_ID))
m.get_root().html.add_child(folium.Element(_panel))
print("Panel legible IPM/ICS inyectado (medidores + lectura + privaciones).")

## 7. Exportacion del HTML

In [ ]:
agregar_export(m, "IPM y ICS por manzana", "lab_itt_ipm")
html_lab=os.path.join(MAPAS_DIR,"lab_itt_ipm.html")
m.save(html_lab)
print("Mapa laboratorio exportado:", html_lab)
m